<a href="https://colab.research.google.com/github/sandydiamantino/retailrocket-topsis/blob/main/enmc_topsis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avaliação de recomendação com grafo, baseline e TOPSIS

## 1. Apresentação

Este notebook apresenta um experimento de recomendação de produtos baseado em interações de usuários. O estudo utiliza dados do Retailrocket para construir um grafo bipartido entre visitantes e produtos, gerar um ranking baseline e aplicar o método multicritério TOPSIS.

O notebook pode ser executado em dois modos:

- `sample`: utiliza a amostra menor dos dados, adequada para testes e validação do código;
- `full`: utiliza o dataset completo, permitindo uma análise em maior escala.

Independentemente do modo escolhido, o mesmo método é aplicado aos dados por meio da variável `events_active`.

## 2. Objetivos

Os objetivos principais são:

1. carregar e padronizar os dados de eventos;
2. construir uma representação em grafo das relações entre visitantes e produtos;
3. calcular métricas de importância dos produtos;
4. construir um ranking baseline;
5. construir um ranking multicritério usando TOPSIS;
6. comparar os rankings produzidos;
7. avaliar a capacidade dos métodos de antecipar interações futuras.

## 3. Organização dos dados

Os dados principais utilizados são:

- `events.csv`: contém os eventos dos visitantes, como visualizações, adições ao carrinho e transações;
- `item_properties.csv`: contém propriedades associadas aos produtos.

Após o carregamento, os dados são organizados nas seguintes variáveis:

- `events_sample`: amostra dos eventos;
- `events_full`: dataset completo dos eventos;
- `events_active`: dataset selecionado para a execução atual;
- `properties_active`: propriedades correspondentes ao modo selecionado.

A escolha entre os modos é feita pela variável:

```python
DATA_MODE = 'sample'
```

ou:

```python
DATA_MODE = 'full'
```

## 4. Padronização dos dados

Antes da modelagem, os dados são verificados e padronizados. Essa etapa inclui:

- verificação dos nomes das colunas;
- conversão dos identificadores para formato numérico;
- padronização dos nomes dos eventos;
- criação da coluna `datetime`;
- identificação de valores ausentes;
- remoção apenas de registros sem informações essenciais;
- ordenação cronológica dos eventos.

A ausência de `transactionid` não é tratada como erro, pois visualizações e adições ao carrinho normalmente não possuem uma transação associada.

## 5. Avaliação temporal

Para evitar o uso de informações futuras na construção dos rankings, os eventos são divididos cronologicamente em:

- `events_train`: eventos utilizados para construir o grafo, o baseline e o TOPSIS;
- `events_test`: eventos futuros utilizados para avaliar os resultados.

A divisão temporal é aplicada tanto ao modo `sample` quanto ao modo `full`.

Essa separação permite verificar se os produtos recomendados no período de treino aparecem posteriormente nas interações observadas no período de teste.

## 6. Construção do grafo

O grafo é representado como uma rede bipartida formada por:

- nós de visitantes;
- nós de produtos;
- arestas representando interações entre visitantes e produtos.

As interações recebem pesos diferentes conforme o tipo de evento:

- visualização;
- adição ao carrinho;
- transação.

A representação esparsa é utilizada para reduzir o consumo de memória e permitir o processamento de conjuntos de dados maiores.

## 7. Critérios do TOPSIS

O ranking TOPSIS utiliza cinco critérios, todos classificados como critérios de benefício:

| Critério | Origem | Interpretação |
|---|---|---|
| Pontuação de coocorrência | Baseline | Intensidade das relações de coocorrência dos produtos |
| PageRank | Grafo | Importância estrutural do produto na rede |
| Grau ponderado | Grafo | Intensidade das interações associadas ao produto |
| Recência | Eventos | Atualidade das interações do produto |

Quanto maior o valor de cada critério, melhor é o desempenho do produto segundo a respectiva dimensão.

## 8. Baseline

O baseline representa uma estratégia de comparação baseada na popularidade dos produtos. Ele serve como referência para verificar se o TOPSIS produz resultados diferentes ou melhores.

O baseline não utiliza a combinação multicritério do TOPSIS. Seu objetivo é representar uma estratégia mais simples, baseada no comportamento observado nos dados de treino.

## 9. TOPSIS

O TOPSIS combina os cinco critérios em uma pontuação única. O método compara cada produto com uma solução ideal e uma solução anti-ideal.

Produtos mais próximos da solução ideal recebem melhores posições no ranking.

O resultado final é uma lista ordenada de produtos acompanhada de:

- posição no ranking;
- pontuação TOPSIS;
- valores dos critérios utilizados.

## 10. Comparação dos métodos

Os rankings são comparados por meio de:

- sobreposição entre os Top 5, Top 10, Top 20, Top 50 e Top 100;
- índice de Jaccard;
- correlação de Spearman;
- correlação de Kendall;
- análise dos produtos que subiram ou caíram de posição;
- identificação dos produtos exclusivos de cada método.

Essas medidas mostram o grau de concordância entre o baseline e o TOPSIS, mas não são suficientes, isoladamente, para determinar qual método é mais eficaz.

## 11. Avaliação da eficácia

A eficácia dos métodos é avaliada usando os eventos futuros de `events_test`.

As principais métricas consideradas são:

- Hit Rate@K;
- Precision@K;
- Recall@K;
- MRR;
- NDCG@K.

Um método será considerado mais eficaz quando conseguir antecipar melhor os produtos que aparecem nas interações futuras, especialmente em transações.

## 12. Reprodutibilidade

O notebook foi estruturado para permitir a execução do mesmo pipeline nos modos `sample` e `full`.

A alteração do modo ocorre somente no início:

```python
DATA_MODE = 'sample'
```

ou:

```python
DATA_MODE = 'full'
```

O restante do processamento utiliza as mesmas etapas, critérios, pesos e métricas de avaliação. Dessa forma, as diferenças observadas entre os resultados podem ser atribuídas principalmente à escala e à composição dos dados, e não à aplicação de metodologias diferentes.

### Configuração do ambiente de execução

In [1]:
# Importando as bibliotecas

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import zipfile
import shutil

from scipy.sparse import coo_matrix
from scipy.stats import spearmanr, kendalltau

from IPython.display import display


# Configurações de visualização

pd.set_option(
    'display.max_columns',
    None
)

pd.set_option(
    'display.max_rows',
    100
)

sns.set_theme(
    style='whitegrid'
)


print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


### Seleção do modo de execução e carregamento dos dados

In [2]:
# Escolha do modo de execução

DATA_MODE = 'full'
# Opções válidas:
# 'sample' → usa os arquivos menores da amostra
# 'full'   → usa o dataset completo extraído do ZIP


if DATA_MODE not in ['sample', 'full']:
    raise ValueError(
        "DATA_MODE deve ser 'sample' ou 'full'."
    )


print('Modo selecionado:', DATA_MODE)

Modo selecionado: full


In [3]:
# Carregando os dados conforme o modo selecionado

if DATA_MODE == 'sample':

    EVENTS_URL = ('https://raw.githubusercontent.com/sandydiamantino/retailrocket-topsis/refs/heads/main/data/sample_events.csv')
    PROPERTIES_URL = ('https://raw.githubusercontent.com/sandydiamantino/retailrocket-topsis/refs/heads/main/data/sample_item_properties.csv')

    events_sample = pd.read_csv(
        EVENTS_URL
    )

    properties_sample = pd.read_csv(
        PROPERTIES_URL
    )

    events_active = events_sample.copy()

    properties_active = properties_sample.copy()

    print('Amostra carregada.')

elif DATA_MODE == 'full':

    # Localizando o arquivo ZIP no ambiente

    zip_candidates = list(
        Path('/content').glob('*.zip')
    )

    if len(zip_candidates) == 0:
        raise FileNotFoundError(
            'Nenhum arquivo ZIP foi encontrado em /content.'
        )

    zip_path = zip_candidates[0]

    extract_dir = Path(
        '/content/retailrocket_full'
    )

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Descompactando o ZIP

    with zipfile.ZipFile(
        zip_path,
        'r'
    ) as zip_ref:
        zip_ref.extractall(extract_dir)

    # Localizando events.csv

    events_files = list(
        extract_dir.rglob('events.csv')
    )

    if len(events_files) == 0:
        raise FileNotFoundError(
            'events.csv não foi encontrado.'
        )

    full_events_path = events_files[0]

    # Carregando events.csv completo

    events_full = pd.read_csv(
        full_events_path,
        low_memory=False
    )

    # Localizando as partes de properties

    properties_files = sorted(
        extract_dir.rglob(
            'item_properties_part*.csv'
        )
    )

    if len(properties_files) == 0:
        raise FileNotFoundError(
            'Nenhum arquivo de propriedades foi encontrado.'
        )

    # Unindo as partes de properties

    properties_full = pd.concat(
        [
            pd.read_csv(
                file_path,
                low_memory=False
            )
            for file_path in properties_files
        ],
        ignore_index=True
    )

    events_active = events_full.copy()

    properties_active = properties_full.copy()

    print('Dataset completo carregado.')
    print('Arquivo de eventos:', full_events_path)
    print(
        'Partes de propriedades:',
        len(properties_files)
    )

else:

    raise ValueError(
        "DATA_MODE deve ser 'sample' ou 'full'."
    )

print(
    'Modo ativo:',
    DATA_MODE
)

print(
    'Eventos carregados:',
    f'{len(events_active):,}'
)

print(
    'Registros de propriedades:',
    f'{len(properties_active):,}'
)

Dataset completo carregado.
Arquivo de eventos: /content/retailrocket_full/retailrocket/events.csv
Partes de propriedades: 2
Modo ativo: full
Eventos carregados: 2,756,101
Registros de propriedades: 20,275,902


In [4]:
# Verificando o tamanho das tabelas
print('Tabela de eventos:')
print(f'Linhas: {events_active.shape[0]:,}')
print(f'Colunas: {events_active.shape[1]}')

print('\nTabela de propriedades:')
print(f'Linhas: {properties_active.shape[0]:,}')
print(f'Colunas: {properties_active.shape[1]}')

Tabela de eventos:
Linhas: 2,756,101
Colunas: 5

Tabela de propriedades:
Linhas: 20,275,902
Colunas: 4


### Configuração dos perfis de execução para configuração dos pesos do TOPSIS

### Seleção do perfil de pesos do TOPSIS

O método TOPSIS utiliza cinco critérios para classificar os produtos:

1. pontuação de coocorrência;
2. PageRank;
3. grau ponderado;
4. recência.

Para tornar o experimento reproduzível, os pesos são organizados em perfis predefinidos. A seleção do perfil é realizada pela variável `WEIGHT_PROFILE`, localizada no início do notebook.

O perfil `equal` atribui o mesmo peso a todos os critérios:

| Critério | Peso |
|---|---:|
| Pontuação de coocorrência | 20% |
| PageRank | 20% |
| Grau ponderado | 20% |
| Recência | 20% |

O perfil `cooccurrence_focus` atribui maior peso à pontuação de coocorrência:

| Critério | Peso |
|---|---:|
| Pontuação de coocorrência | 40% |
| PageRank | 20% |
| Grau ponderado | 20% |
| Recência | 20% |

A seleção do perfil não deve ser feita utilizando diretamente o conjunto `events_test`. Esse conjunto deve permanecer reservado para a avaliação final, evitando vazamento de informação. Caso seja necessário escolher o perfil com base no desempenho, deve-se utilizar um conjunto de validação separado dentro dos dados de treino.

In [5]:
# Configuração principal do TOPSIS

TOPSIS_CRITERIA = [
    'cooccurrence_score',
    'pagerank',
    'weighted_degree',
    'recency_score'
]

WEIGHT_PROFILES = {
    'equal': np.array([
        0.25,
        0.25,
        0.25,
        0.25
    ]),

    'cooccurrence_focus': np.array([
        0.40,
        0.20,
        0.20,
        0.20
    ])
}

WEIGHT_PROFILE = 'cooccurrence_focus'

if WEIGHT_PROFILE not in WEIGHT_PROFILES:
    raise ValueError(
        'Perfil de pesos inválido.'
    )

topsis_weights = (
    WEIGHT_PROFILES[WEIGHT_PROFILE]
)

if not np.isclose(
    topsis_weights.sum(),
    1.0
):
    raise ValueError(
        'A soma dos pesos deve ser igual a 1.'
    )

print(
    'Perfil selecionado:',
    WEIGHT_PROFILE
)

print(
    'Critérios utilizados:',
    dict(
        zip(
            TOPSIS_CRITERIA,
            topsis_weights
        )
    )
)

Perfil selecionado: cooccurrence_focus
Critérios utilizados: {'cooccurrence_score': np.float64(0.4), 'pagerank': np.float64(0.2), 'weighted_degree': np.float64(0.2), 'recency_score': np.float64(0.2)}


### Padronização dos dados

In [6]:
# Padronizando os dados ativos

events_active = events_active.copy()
properties_active = properties_active.copy()


# 1. Verificar colunas disponíveis

print('Colunas de events_active:')
print(events_active.columns.tolist())

print('\nColunas de properties_active:')
print(properties_active.columns.tolist())


# 2. Verificar colunas obrigatórias

required_event_columns = [
    'timestamp',
    'visitorid',
    'event',
    'itemid'
]

missing_event_columns = [
    column
    for column in required_event_columns
    if column not in events_active.columns
]

if missing_event_columns:
    raise KeyError(
        'Colunas ausentes em events_active: '
        + str(missing_event_columns)
    )


required_property_columns = [
    'timestamp',
    'itemid',
    'property',
    'value'
]

missing_property_columns = [
    column
    for column in required_property_columns
    if column not in properties_active.columns
]

if missing_property_columns:
    raise KeyError(
        'Colunas ausentes em properties_active: '
        + str(missing_property_columns)
    )


# 3. Padronizar identificadores dos eventos

events_active['visitorid'] = pd.to_numeric(
    events_active['visitorid'],
    errors='coerce'
)

events_active['itemid'] = pd.to_numeric(
    events_active['itemid'],
    errors='coerce'
)


# 4. Padronizar o tipo de evento

events_active['event'] = (
    events_active['event']
    .astype('string')
    .str.strip()
    .str.lower()
)


# 5. Criar a coluna datetime

events_active['datetime'] = pd.to_datetime(
    events_active['timestamp'],
    unit='ms',
    errors='coerce'
)


# 6. Verificar valores ausentes antes da limpeza

print('\nValores ausentes em events_active:')
display(
    events_active.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('quantidade')
)


# 7. Remover somente linhas sem informações essenciais

events_active = events_active.dropna(
    subset=[
        'visitorid',
        'itemid',
        'event',
        'datetime'
    ]
).copy()


# 8. Padronizar propriedades

properties_active['itemid'] = pd.to_numeric(
    properties_active['itemid'],
    errors='coerce'
)

properties_active['property'] = (
    properties_active['property']
    .astype('string')
    .str.strip()
    .str.lower()
)

properties_active['properties_datetime'] = pd.to_datetime(
    properties_active['timestamp'],
    unit='ms',
    errors='coerce'
)


# 9. Remover apenas propriedades sem identificação utilizável

properties_active = properties_active.dropna(
    subset=[
        'itemid',
        'property'
    ]
).copy()


# 10. Ordenar os dados por data

events_active = events_active.sort_values(
    by='datetime'
).reset_index(drop=True)

properties_active = properties_active.sort_values(
    by='properties_datetime'
).reset_index(drop=True)


# 11. Exibir resumo final

print('\nPadronização concluída.')
print(
    'Eventos ativos:',
    f'{len(events_active):,}'
)

print(
    'Propriedades ativas:',
    f'{len(properties_active):,}'
)

print(
    'Visitantes:',
    f"{events_active['visitorid'].nunique():,}"
)

print(
    'Produtos:',
    f"{events_active['itemid'].nunique():,}"
)

print(
    'Tipos de evento:',
    events_active['event'].unique()
)

print(
    'Data inicial:',
    events_active['datetime'].min()
)

print(
    'Data final:',
    events_active['datetime'].max()
)



Colunas de events_active:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']

Colunas de properties_active:
['timestamp', 'itemid', 'property', 'value']

Valores ausentes em events_active:


,quantidade
transactionid,2733644
timestamp,0
visitorid,0
event,0
itemid,0
datetime,0



Padronização concluída.
Eventos ativos: 2,756,101
Propriedades ativas: 20,275,902
Visitantes: 1,407,580
Produtos: 235,061
Tipos de evento: <StringArray>
['addtocart', 'view', 'transaction']
Length: 3, dtype: string
Data inicial: 2015-05-03 03:00:04.384000
Data final: 2015-09-18 02:59:47.788000


### Divisão temporal para avaliação fora da amostra

Executada para o modo selecionado (`sample` ou `full`), separando os eventos  cronologicamente em `events_train` e `events_test`.

O modelo será construído usando `events_train` e avaliado posteriormente usando `events_test`.

In [7]:
# Realizando a divisão temporal para avaliação fora da amostra

events_evaluation = (
    events_active
    .sort_values(by='datetime')
    .reset_index(drop=True)
    .copy()
)

# Usa os primeiros 80% dos eventos para treino
cutoff_date = events_evaluation[
    'datetime'
].quantile(0.80)

events_train = events_evaluation[
    events_evaluation['datetime'] <= cutoff_date
].copy()

events_test = events_evaluation[
    events_evaluation['datetime'] > cutoff_date
].copy()

# Reinicia os índices
events_train = events_train.reset_index(
    drop=True
)

events_test = events_test.reset_index(
    drop=True
)

print('Divisão temporal concluída.')
print('Data de corte:', cutoff_date)

print(
    'Eventos de treino:',
    f'{len(events_train):,}'
)

print(
    'Eventos de teste:',
    f'{len(events_test):,}'
)

print(
    'Transações no treino:',
    f"{(events_train['event'] == 'transaction').sum():,}"
)

print(
    'Transações no teste:',
    f"{(events_test['event'] == 'transaction').sum():,}"
)

Divisão temporal concluída.
Data de corte: 2015-08-18 04:23:31.966000128
Eventos de treino: 2,204,881
Eventos de teste: 551,220
Transações no treino: 17,864
Transações no teste: 4,593


In [8]:
# Verificação adicional
print(
    'Início do treino:',
    events_train['datetime'].min()
)

print(
    'Fim do treino:',
    events_train['datetime'].max()
)

print(
    'Início do teste:',
    events_test['datetime'].min()
)

print(
    'Fim do teste:',
    events_test['datetime'].max()
)

Início do treino: 2015-05-03 03:00:04.384000
Fim do treino: 2015-08-18 04:23:31.966000
Início do teste: 2015-08-18 04:23:37.919000
Fim do teste: 2015-09-18 02:59:47.788000


### Criação das pastas nas quais serão salvos os resultados

In [9]:
# Criando pastas para resultados de treino e teste

output_dir = Path('output')

train_output_dir = (
    output_dir / 'train'
)

test_output_dir = (
    output_dir / 'test'
)

train_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print('Pastas criadas:')
print(train_output_dir)
print(test_output_dir)

Pastas criadas:
output/train
output/test


### Construção do grafo com o conjunto `events_train`

In [10]:
# Preparando as interações do conjunto de treino

events_train_graph = events_train[
    [
        'visitorid',
        'itemid',
        'event',
        'datetime'
    ]
].dropna(
    subset=[
        'visitorid',
        'itemid',
        'event'
    ]
).copy()

event_weights = {
    'view': 1.0,
    'addtocart': 3.0,
    'transaction': 5.0
}

events_train_graph['interaction_weight'] = (
    events_train_graph['event']
    .map(event_weights)
    .fillna(0.0)
)

events_train_graph = events_train_graph[
    events_train_graph['interaction_weight'] > 0
].copy()

print(
    'Interações de treino:',
    f'{len(events_train_graph):,}'
)

display(
    events_train_graph['event']
    .value_counts()
    .to_frame('quantidade')
)

Interações de treino: 2,204,881


,quantidade
event,
view,2132032
addtocart,54985
transaction,17864


In [11]:
# Agregando relações entre visitantes e produtos

user_item_train = (
    events_train_graph
    .groupby(
        [
            'visitorid',
            'itemid'
        ],
        as_index=False
    )
    .agg(
        interaction_weight=(
            'interaction_weight',
            'sum'
        ),
        event_count=(
            'event',
            'size'
        ),
        last_datetime=(
            'datetime',
            'max'
        )
    )
)

print(
    'Relações únicas visitante-produto:',
    f'{len(user_item_train):,}'
)

display(
    user_item_train.head()
)

Relações únicas visitante-produto: 1,713,175


,visitorid,itemid,interaction_weight,event_count,last_datetime
0,1,72028,1.0,1,2015-08-13 17:46:06.444
1,2,216305,2.0,2,2015-08-07 18:17:43.170
2,2,259884,1.0,1,2015-08-07 17:56:52.664
3,2,325215,3.0,3,2015-08-07 18:20:57.845
4,2,342816,2.0,2,2015-08-07 18:17:24.375


In [12]:
# Construindo o grafo bipartido visitante-produto

user_codes_train, user_ids_train = pd.factorize(
    user_item_train['visitorid']
)

item_codes_train, item_ids_train = pd.factorize(
    user_item_train['itemid']
)

train_graph_matrix = coo_matrix(
    (
        user_item_train[
            'interaction_weight'
        ].to_numpy(),
        (
            user_codes_train,
            item_codes_train
        )
    ),
    shape=(
        len(user_ids_train),
        len(item_ids_train)
    )
).tocsr()

print('Grafo de treino criado.')
print(
    'Visitantes:',
    f'{train_graph_matrix.shape[0]:,}'
)
print(
    'Produtos:',
    f'{train_graph_matrix.shape[1]:,}'
)
print(
    'Arestas:',
    f'{train_graph_matrix.nnz:,}'
)

Grafo de treino criado.
Visitantes: 1,123,767
Produtos: 212,916
Arestas: 1,713,175


### Cálculo das métricas para o conjunto `elements_train`

In [13]:
# Calculando GRAU

degree_train = np.asarray(
    train_graph_matrix.getnnz(axis=0)
).ravel()

weighted_degree_train = np.asarray(
    train_graph_matrix.sum(axis=0)
).ravel()

graph_metrics_train = pd.DataFrame({
    'itemid': item_ids_train,
    'degree': degree_train,
    'weighted_degree': weighted_degree_train
})

print('Grau dos produtos calculado.')

display(
    graph_metrics_train
    .sort_values(
        by='weighted_degree',
        ascending=False
    )
    .head(20)
)

Grau dos produtos calculado.


,itemid,degree,weighted_degree
1064,461686,966,2827.0
52,5411,1920,2165.0
941,257040,972,1899.0
480,187946,1553,1831.0
522,309778,1046,1772.0
626,370653,1373,1625.0
756,7943,687,1576.0
213,369447,607,1488.0
1782,298009,1183,1453.0
525,48030,614,1443.0


In [14]:
# Calculando PAGERANK

damping = 0.85
max_iterations = 100
tolerance = 1e-10

num_users, num_items = (
    train_graph_matrix.shape
)

num_nodes = num_users + num_items

user_weighted_degree = np.asarray(
    train_graph_matrix.sum(axis=1)
).ravel()

item_weighted_degree = np.asarray(
    train_graph_matrix.sum(axis=0)
).ravel()

user_weighted_degree[
    user_weighted_degree == 0
] = 1

item_weighted_degree[
    item_weighted_degree == 0
] = 1

pagerank_users = np.full(
    num_users,
    1 / num_nodes
)

pagerank_items = np.full(
    num_items,
    1 / num_nodes
)

teleport = (
    1 - damping
) / num_nodes

for iteration in range(max_iterations):

    new_pagerank_users = (
        teleport
        + damping
        * (
            train_graph_matrix
            @ (
                pagerank_items
                / item_weighted_degree
            )
        )
    )

    new_pagerank_items = (
        teleport
        + damping
        * (
            train_graph_matrix.T
            @ (
                pagerank_users
                / user_weighted_degree
            )
        )
    )

    error = (
        np.abs(
            new_pagerank_users
            - pagerank_users
        ).sum()
        + np.abs(
            new_pagerank_items
            - pagerank_items
        ).sum()
    )

    pagerank_users = new_pagerank_users
    pagerank_items = new_pagerank_items

    if error < tolerance:
        break

graph_metrics_train['pagerank'] = (
    pagerank_items
)

print(
    'PageRank calculado em',
    iteration + 1,
    'iterações.'
)

display(
    graph_metrics_train
    .sort_values(
        by='pagerank',
        ascending=False
    )
    .head(20)
)

PageRank calculado em 100 iterações.


,itemid,degree,weighted_degree,pagerank
480,187946,1553,1831.0,0.000502
52,5411,1920,2165.0,0.000500
626,370653,1373,1625.0,0.000395
1782,298009,1183,1453.0,0.000322
293,335975,1049,1289.0,0.000293
1064,461686,966,2827.0,0.000271
1434,96924,1070,1272.0,0.000270
7839,151444,851,944.0,0.000255
4503,441668,820,1179.0,0.000215
522,309778,1046,1772.0,0.000208


In [15]:
# Calculando COOCORRÊNCIA

user_item_cooccurrence = (
    user_item_train.copy()
)

# Quantidade de produtos distintos associados a cada visitante
user_item_cooccurrence[
    'user_item_count'
] = (
    user_item_cooccurrence
    .groupby('visitorid')['itemid']
    .transform('count')
)

# Quantidade de outros produtos associados ao mesmo visitante
user_item_cooccurrence[
    'other_items_count'
] = (
    user_item_cooccurrence['user_item_count']
    - 1
)

# Intensidade total das interações dos visitantes
user_total_weight = (
    user_item_cooccurrence
    .groupby('visitorid')['interaction_weight']
    .transform('sum')
)

# Peso das interações dos outros produtos
user_item_cooccurrence[
    'other_items_weight'
] = (
    user_total_weight
    - user_item_cooccurrence[
        'interaction_weight'
    ]
)

cooccurrence_train = (
    user_item_cooccurrence
    .groupby('itemid', as_index=False)
    .agg(
        cooccurrence_count=(
            'other_items_count',
            'sum'
        ),
        cooccurrence_score=(
            'other_items_weight',
            'sum'
        )
    )
)

print('Coocorrência calculada.')

display(
    cooccurrence_train
    .sort_values(
        by='cooccurrence_score',
        ascending=False
    )
    .head(20)
)

Coocorrência calculada.


,itemid,cooccurrence_count,cooccurrence_score
54883,119736,38314,94482.0
168600,369158,30791,74465.0
210608,461686,28251,67426.0
208051,456056,27969,66562.0
16902,37029,28269,66108.0
97648,213834,26461,64338.0
137738,301721,26016,63762.0
200750,439963,26076,63742.0
85282,186702,24464,61630.0
7884,17478,25095,61393.0


In [16]:
# Calculando RECÊNCIA

last_interaction_train = (
    events_train_graph
    .groupby('itemid', as_index=False)
    .agg(
        last_interaction=(
            'datetime',
            'max'
        )
    )
)

cutoff_train = (
    events_train['datetime'].max()
)

last_interaction_train[
    'days_since_interaction'
] = (
    cutoff_train
    - last_interaction_train[
        'last_interaction'
    ]
).dt.total_seconds() / 86400

last_interaction_train[
    'recency_score'
] = 1 / (
    1
    + last_interaction_train[
        'days_since_interaction'
    ]
)

print('Recência calculada.')

display(
    last_interaction_train
    .sort_values(
        by='recency_score',
        ascending=False
    )
    .head(20)
)

Recência calculada.


,itemid,last_interaction,days_since_interaction,recency_score
139131,304858,2015-08-18 04:23:31.966,0.000000,1.000000
97925,214436,2015-08-18 04:23:22.451,0.000110,0.999890
39600,86466,2015-08-18 04:23:22.342,0.000111,0.999889
201201,440937,2015-08-18 04:23:20.445,0.000133,0.999867
88227,193205,2015-08-18 04:23:01.129,0.000357,0.999643
149410,327591,2015-08-18 04:22:55.986,0.000416,0.999584
204273,447661,2015-08-18 04:22:44.694,0.000547,0.999453
104740,229428,2015-08-18 04:22:42.875,0.000568,0.999432
25647,55936,2015-08-18 04:22:41.548,0.000584,0.999417
69756,152433,2015-08-18 04:22:38.217,0.000622,0.999378


In [17]:
# Calculando TAXA DE CONVERSÃO

unique_visitors_train = (
    events_train_graph
    .groupby('itemid')['visitorid']
    .nunique()
    .rename('unique_visitors')
)

unique_buyers_train = (
    events_train_graph[
        events_train_graph['event']
        == 'transaction'
    ]
    .groupby('itemid')['visitorid']
    .nunique()
    .rename('unique_buyers')
)

conversion_train = pd.concat(
    [
        unique_visitors_train,
        unique_buyers_train
    ],
    axis=1
).fillna(0).reset_index()

conversion_train[
    'conversion_rate'
] = (
    conversion_train['unique_buyers']
    / conversion_train['unique_visitors']
    .replace(0, np.nan)
)

conversion_train[
    'conversion_rate'
] = (
    conversion_train['conversion_rate']
    .fillna(0.0)
)

print('Taxa de conversão calculada.')

display(
    conversion_train
    .sort_values(
        by='conversion_rate',
        ascending=False
    )
    .head(20)
)

Taxa de conversão calculada.


,itemid,unique_visitors,unique_buyers,conversion_rate
19799,43354,1,1.0,1.0
144353,316239,1,1.0,1.0
102298,224166,1,1.0,1.0
124933,273652,1,1.0,1.0
198243,434521,1,1.0,1.0
125414,274731,1,1.0,1.0
35552,77640,1,1.0,1.0
100384,219849,1,1.0,1.0
188177,412137,1,1.0,1.0
184426,403933,1,1.0,1.0


### Consolidação dos critérios TOPSIS

In [18]:
# Consolidando os cinco critérios do TOPSIS

item_metrics_train_final = (
    graph_metrics_train[
        [
            'itemid',
            'pagerank',
            'weighted_degree'
        ]
    ]
    .merge(
        cooccurrence_train[
            [
                'itemid',
                'cooccurrence_score'
            ]
        ],
        on='itemid',
        how='left'
    )
    .merge(
        last_interaction_train[
            [
                'itemid',
                'recency_score'
            ]
        ],
        on='itemid',
        how='left'
    )
    .merge(
        conversion_train[
            [
                'itemid',
                'conversion_rate'
            ]
        ],
        on='itemid',
        how='left'
    )
)

item_metrics_train_final = (
    item_metrics_train_final
    .fillna(0)
)

criteria_columns = [
    'cooccurrence_score',
    'pagerank',
    'weighted_degree',
    'recency_score',
    'conversion_rate'
]

print('Critérios consolidados:')
print(criteria_columns)

print(
    'Produtos avaliados:',
    f'{len(item_metrics_train_final):,}'
)

display(
    item_metrics_train_final.head()
)

Critérios consolidados:
['cooccurrence_score', 'pagerank', 'weighted_degree', 'recency_score', 'conversion_rate']
Produtos avaliados: 212,916


,itemid,pagerank,weighted_degree,cooccurrence_score,recency_score,conversion_rate
0,72028,0.000005,30.0,48.0,0.231458,0.000000
1,216305,0.000096,556.0,9456.0,0.960321,0.002427
2,259884,0.000056,395.0,12564.0,0.769584,0.017167
3,325215,0.000019,113.0,4515.0,0.233067,0.025000
4,342816,0.000014,132.0,15139.0,0.673662,0.056338


In [19]:
# Verificando os critérios antes do TOPSIS

criteria_summary = (
    item_metrics_train_final[
        criteria_columns
    ]
    .describe()
    .T
)

display(criteria_summary)

print('Valores ausentes:')
display(
    item_metrics_train_final[
        criteria_columns
    ]
    .isna()
    .sum()
)

print('Valores infinitos:')
display(
    np.isinf(
        item_metrics_train_final[
            criteria_columns
        ]
        .to_numpy()
    )
    .sum(axis=0)
)

,count,mean,std,min,25%,50%,75%,max
cooccurrence_score,212916.0,558.724070,2479.984134,0.000000e+00,0.000000e+00,2.000000e+00,31.000000,94482.000000
pagerank,212916.0,0.000002,0.000006,1.354114e-07,7.481205e-07,9.475427e-07,0.000002,0.000502
weighted_degree,212916.0,11.207739,35.143769,1.000000e+00,1.000000e+00,3.000000e+00,8.000000,2827.000000
recency_score,212916.0,0.115605,0.191804,9.254453e-03,1.811257e-02,3.803713e-02,0.109472,1.000000
conversion_rate,212916.0,0.004215,0.030434,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1.000000


Valores ausentes:


,0
cooccurrence_score,0
pagerank,0
weighted_degree,0
recency_score,0
conversion_rate,0


Valores infinitos:


array([0, 0, 0, 0, 0])

### *Save* dos resultados do treino

In [20]:
# Salvando métricas e estruturas do treino

graph_metrics_train.to_csv(
    train_output_dir
    / 'graph_metrics_train.csv',
    index=False
)

cooccurrence_train.to_csv(
    train_output_dir
    / 'cooccurrence_train.csv',
    index=False
)

last_interaction_train.to_csv(
    train_output_dir
    / 'recency_train.csv',
    index=False
)

conversion_train.to_csv(
    train_output_dir
    / 'conversion_train.csv',
    index=False
)

item_metrics_train_final.to_csv(
    train_output_dir
    / 'item_metrics_train_final.csv',
    index=False
)


user_item_train.to_csv(
    train_output_dir
    / 'user_item_train.csv',
    index=False
)

print(
    'Métricas consolidadas salvas em:',
    train_output_dir
    / 'item_metrics_train_final.csv'
)

print(
    'Resultados de treino salvos em:',
    train_output_dir
    / 'user_item_train.csv'
)

Métricas consolidadas salvas em: output/train/item_metrics_train_final.csv
Resultados de treino salvos em: output/train/user_item_train.csv


### Construção do Baseline e TOPSIS

In [21]:
# Construindo o baseline de treino

baseline_results_train = (
    item_metrics_train_final
    .sort_values(
        by=[
            'cooccurrence_score',
            'weighted_degree',
            'pagerank',
            'recency_score',
            'conversion_rate'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

baseline_results_train[
    'baseline_rank'
] = (
    baseline_results_train.index + 1
)

print('Top 20 do baseline de treino:')

display(
    baseline_results_train[
        [
            'baseline_rank',
            'itemid',
            'cooccurrence_score',
            'weighted_degree',
            'pagerank',
            'recency_score',
            'conversion_rate'
        ]
    ].head(20)
)

Top 20 do baseline de treino:


,baseline_rank,itemid,cooccurrence_score,weighted_degree,pagerank,recency_score,conversion_rate
0,1,119736,94482.0,1081.0,0.000099,0.878456,0.176471
1,2,369158,74465.0,560.0,0.000056,0.953668,0.048443
2,3,461686,67426.0,2827.0,0.000271,0.957749,0.094203
3,4,456056,66562.0,251.0,0.000023,0.834215,0.117647
4,5,37029,66108.0,1241.0,0.000154,0.989801,0.031065
5,6,213834,64338.0,515.0,0.000048,0.803756,0.292398
6,7,301721,63762.0,361.0,0.000036,0.711468,0.059459
7,8,439963,63742.0,228.0,0.000021,0.832434,0.028169
8,9,186702,61630.0,226.0,0.000022,0.877861,0.103093
9,10,17478,61393.0,953.0,0.000104,0.818523,0.092958


In [22]:
# Aplicando TOPSIS aos cinco critérios

topsis_criteria = TOPSIS_CRITERIA
topsis_weights = WEIGHT_PROFILES[
    WEIGHT_PROFILE
]

X_train = (
    item_metrics_train_final[
        topsis_criteria
    ]
    .astype(float)
)

# Normalização vetorial

norms = np.sqrt(
    (X_train ** 2).sum(axis=0)
)

norms = norms.replace(
    0,
    1
)

R_train = X_train / norms

# Aplicação dos pesos

V_train = (
    R_train
    * topsis_weights
)

# Soluções ideal e anti-ideal

ideal_solution = V_train.max(
    axis=0
)

anti_ideal_solution = V_train.min(
    axis=0
)

# Distâncias euclidianas
# Os parênteses externos são essenciais

distance_to_ideal = np.sqrt(
    (
        (V_train - ideal_solution) ** 2
    ).sum(axis=1)
)

distance_to_anti_ideal = np.sqrt(
    (
        (V_train - anti_ideal_solution) ** 2
    ).sum(axis=1)
)

# Evita divisão por zero

denominator = (
    distance_to_ideal
    + distance_to_anti_ideal
)

denominator = denominator.replace(
    0,
    1
)

# Pontuação TOPSIS

item_metrics_train_final[
    'topsis_score'
] = (
    distance_to_anti_ideal
    / denominator
)

# Ranking final

topsis_results_train = (
    item_metrics_train_final
    .sort_values(
        by='topsis_score',
        ascending=False
    )
    .reset_index(drop=True)
)

topsis_results_train[
    'topsis_rank'
] = (
    topsis_results_train.index + 1
)

print('TOPSIS calculado com sucesso.')

display(
    topsis_results_train[
        [
            'topsis_rank',
            'itemid',
            'topsis_score',
            'cooccurrence_score',
            'pagerank',
            'weighted_degree',
            'recency_score',
            'conversion_rate'
        ]
    ].head(20)
)

TOPSIS calculado com sucesso.


,topsis_rank,itemid,topsis_score,cooccurrence_score,pagerank,weighted_degree,recency_score,conversion_rate
0,1,461686,0.700094,67426.0,0.000271,2827.0,0.957749,0.094203
1,2,5411,0.599803,10107.0,0.000500,2165.0,0.993131,0.000000
2,3,187946,0.555087,545.0,0.000502,1831.0,0.999366,0.000000
3,4,257040,0.547444,59618.0,0.000195,1899.0,0.979276,0.024691
4,5,309778,0.533881,56200.0,0.000208,1772.0,0.854752,0.014340
5,6,7943,0.516323,58563.0,0.000203,1576.0,0.851892,0.058224
6,7,119736,0.496618,94482.0,0.000099,1081.0,0.878456,0.176471
7,8,370653,0.494608,2065.0,0.000395,1625.0,0.971511,0.000000
8,9,37029,0.469120,66108.0,0.000154,1241.0,0.989801,0.031065
9,10,48030,0.467444,53933.0,0.000175,1443.0,0.954788,0.060261


In [23]:
# Comparando baseline e TOPSIS no conjunto de treino

ranking_train = (
    baseline_results_train[
        [
            'itemid',
            'baseline_rank'
        ]
    ]
    .merge(
        topsis_results_train[
            [
                'itemid',
                'topsis_rank',
                'topsis_score'
            ]
        ],
        on='itemid',
        how='inner'
    )
)

ranking_train[
    'mudanca_posicao'
] = (
    ranking_train['baseline_rank']
    - ranking_train['topsis_rank']
)

ranking_train = (
    ranking_train
    .sort_values(
        by='topsis_rank'
    )
    .reset_index(drop=True)
)

print(
    'Produtos comparados:',
    f'{len(ranking_train):,}'
)

display(
    ranking_train.head(20)
)

Produtos comparados: 212,916


,itemid,baseline_rank,topsis_rank,topsis_score,mudanca_posicao
0,461686,3,1,0.700094,2
1,5411,3235,2,0.599803,3233
2,187946,24310,3,0.555087,24307
3,257040,15,4,0.547444,11
4,309778,20,5,0.533881,15
5,7943,16,6,0.516323,10
6,119736,1,7,0.496618,-6
7,370653,13259,8,0.494608,13251
8,37029,5,9,0.469120,-4
9,48030,22,10,0.467444,12


In [24]:
# Salvando os rankings do treino

baseline_results_train.to_csv(
    train_output_dir
    / 'baseline_results_train.csv',
    index=False
)

topsis_results_train.to_csv(
    train_output_dir
    / 'topsis_results_train.csv',
    index=False
)

ranking_train.to_csv(
    train_output_dir
    / 'ranking_train_comparison.csv',
    index=False
)

print('Resultados salvos em:')
print(
    train_output_dir
    / 'baseline_results_train.csv'
)
print(
    train_output_dir
    / 'topsis_results_train.csv'
)
print(
    train_output_dir
    / 'ranking_train_comparison.csv'
)

Resultados salvos em:
output/train/baseline_results_train.csv
output/train/topsis_results_train.csv
output/train/ranking_train_comparison.csv


### Realizando os testes

In [25]:
# Preparando as transações futuras para avaliação

test_transactions = events_test[
    events_test['event'] == 'transaction'
].copy()

test_transaction_counts = (
    test_transactions
    .groupby('itemid')
    .size()
    .reset_index(
        name='test_transactions'
    )
)

test_unique_buyers = (
    test_transactions
    .groupby('itemid')['visitorid']
    .nunique()
    .reset_index(
        name='test_unique_buyers'
    )
)

test_relevant_items = (
    test_transaction_counts
    .merge(
        test_unique_buyers,
        on='itemid',
        how='left'
    )
)

train_items = set(
    item_metrics_train_final['itemid']
)

test_relevant_items['seen_in_train'] = (
    test_relevant_items['itemid']
    .isin(train_items)
)

print(
    'Transações no teste:',
    f'{len(test_transactions):,}'
)

print(
    'Produtos comprados no teste:',
    f'{len(test_relevant_items):,}'
)

print(
    'Produtos comprados no teste e conhecidos no treino:',
    f"{test_relevant_items['seen_in_train'].sum():,}"
)

print(
    'Produtos novos no teste:',
    f"{(~test_relevant_items['seen_in_train']).sum():,}"
)

Transações no teste: 4,593
Produtos comprados no teste: 3,292
Produtos comprados no teste e conhecidos no treino: 3,091
Produtos novos no teste: 201


In [26]:
# Produtos relevantes que já eram conhecidos no treino

relevant_items = set(
    test_relevant_items.loc[
        test_relevant_items['seen_in_train'],
        'itemid'
    ]
)

print(
    'Itens relevantes para avaliação:',
    f'{len(relevant_items):,}'
)

Itens relevantes para avaliação: 3,091


In [27]:
# Função para avaliar um ranking global

def evaluate_global_ranking(
    ranking_df,
    rank_column,
    relevant_items,
    k_values
):
    results = []

    ranking_items = (
        ranking_df
        .sort_values(
            by=rank_column
        )['itemid']
        .tolist()
    )

    total_relevant = len(
        relevant_items
    )

    for k in k_values:

        recommended_items = set(
            ranking_items[:k]
        )

        hits = (
            recommended_items
            .intersection(relevant_items)
        )

        hit_count = len(hits)

        precision_at_k = (
            hit_count / k
        )

        recall_at_k = (
            hit_count / total_relevant
            if total_relevant > 0
            else 0.0
        )

        hit_at_k = int(
            hit_count > 0
        )

        results.append({
            'k': k,
            'hits': hit_count,
            'hit_at_k': hit_at_k,
            'precision_at_k': precision_at_k,
            'recall_at_k': recall_at_k
        })

    return pd.DataFrame(results)

In [28]:
# Avaliando baseline e TOPSIS no conjunto de teste

k_values = [
    5,
    10,
    20,
    50,
    100
]

baseline_test_evaluation = (
    evaluate_global_ranking(
        ranking_df=baseline_results_train,
        rank_column='baseline_rank',
        relevant_items=relevant_items,
        k_values=k_values
    )
)

baseline_test_evaluation[
    'method'
] = 'Baseline'

topsis_test_evaluation = (
    evaluate_global_ranking(
        ranking_df=topsis_results_train,
        rank_column='topsis_rank',
        relevant_items=relevant_items,
        k_values=k_values
    )
)

topsis_test_evaluation[
    'method'
] = 'TOPSIS'

evaluation_results = pd.concat(
    [
        baseline_test_evaluation,
        topsis_test_evaluation
    ],
    ignore_index=True
)

evaluation_results = evaluation_results[
    [
        'method',
        'k',
        'hits',
        'hit_at_k',
        'precision_at_k',
        'recall_at_k'
    ]
]

display(evaluation_results)

,method,k,hits,hit_at_k,precision_at_k,recall_at_k
0,Baseline,5,5,1,1.00,0.001618
1,Baseline,10,9,1,0.90,0.002912
2,Baseline,20,15,1,0.75,0.004853
3,Baseline,50,33,1,0.66,0.010676
4,Baseline,100,56,1,0.56,0.018117
5,TOPSIS,5,1,1,0.20,0.000324
6,TOPSIS,10,5,1,0.50,0.001618
7,TOPSIS,20,12,1,0.60,0.003882
8,TOPSIS,50,32,1,0.64,0.010353
9,TOPSIS,100,54,1,0.54,0.017470


In [29]:
# Comparação direta entre baseline e TOPSIS

evaluation_comparison = (
    evaluation_results
    .pivot(
        index='k',
        columns='method',
        values=[
            'hits',
            'precision_at_k',
            'recall_at_k'
        ]
    )
    .reset_index()
)

display(evaluation_comparison)

k     hits        precision_at_k        recall_at_k          
method      Baseline TOPSIS       Baseline TOPSIS    Baseline    TOPSIS
0         5      5.0    1.0           1.00   0.20    0.001618  0.000324
1        10      9.0    5.0           0.90   0.50    0.002912  0.001618
2        20     15.0   12.0           0.75   0.60    0.004853  0.003882
3        50     33.0   32.0           0.66   0.64    0.010676  0.010353
4       100     56.0   54.0           0.56   0.54    0.018117  0.017470

A interpretação será:

*   maior hits: mais produtos recomendados apareceram nas transações futuras;
*   maior precision_at_k: maior proporção de acertos dentro do Top-K;
*   maior recall_at_k: maior cobertura dos produtos comprados no teste.








In [30]:
# Diagnóstico dos produtos comprados no teste

print(
    'Transações no teste:',
    len(test_transactions)
)

print(
    'Produtos comprados no teste:',
    len(test_relevant_items)
)

print(
    'Produtos relevantes conhecidos no treino:',
    len(relevant_items)
)

print(
    'Produtos novos no teste:',
    len(
        set(test_relevant_items['itemid'])
        - train_items
    )
)

display(
    test_relevant_items
    .sort_values(
        by='test_transactions',
        ascending=False
    )
)

Transações no teste: 4593
Produtos comprados no teste: 3292
Produtos relevantes conhecidos no treino: 3091
Produtos novos no teste: 201


,itemid,test_transactions,test_unique_buyers,seen_in_train
3255,461686,41,40,True
1471,213834,33,32,True
2354,334401,25,23,True
815,119736,22,5,True
1745,248455,17,13,True
...,...,...,...,...
27,2980,1,1,True
7,856,1,1,True
6,835,1,1,False
5,720,1,1,True


In [31]:
# Posições dos produtos comprados no teste

baseline_rank_lookup = (
    baseline_results_train[
        [
            'itemid',
            'baseline_rank'
        ]
    ]
)

topsis_rank_lookup = (
    topsis_results_train[
        [
            'itemid',
            'topsis_rank'
        ]
    ]
)

test_items_with_ranks = (
    test_relevant_items[
        test_relevant_items['seen_in_train']
    ][
        [
            'itemid',
            'test_transactions',
            'test_unique_buyers'
        ]
    ]
    .merge(
        baseline_rank_lookup,
        on='itemid',
        how='left'
    )
    .merge(
        topsis_rank_lookup,
        on='itemid',
        how='left'
    )
    .sort_values(
        by='test_transactions',
        ascending=False
    )
)

display(test_items_with_ranks)

,itemid,test_transactions,test_unique_buyers,baseline_rank,topsis_rank
3057,461686,41,40,3,1
1387,213834,33,32,6,31
2218,334401,25,23,24,63
764,119736,22,5,1,7
1643,248455,17,13,93,108
...,...,...,...,...,...
1440,220540,1,1,11964,5962
1441,220750,1,1,59711,11365
1442,220793,1,1,29491,62336
1443,220970,1,1,1273,1757


In [32]:
# Avaliação exploratória com Top-K maiores

k_values_extended = [
    100,
    250,
    500,
    1000,
    2000
]

baseline_extended = (
    evaluate_global_ranking(
        ranking_df=baseline_results_train,
        rank_column='baseline_rank',
        relevant_items=relevant_items,
        k_values=k_values_extended
    )
)

baseline_extended['method'] = 'Baseline'

topsis_extended = (
    evaluate_global_ranking(
        ranking_df=topsis_results_train,
        rank_column='topsis_rank',
        relevant_items=relevant_items,
        k_values=k_values_extended
    )
)

topsis_extended['method'] = 'TOPSIS'

evaluation_extended = pd.concat(
    [
        baseline_extended,
        topsis_extended
    ],
    ignore_index=True
)

display(evaluation_extended)

,k,hits,hit_at_k,precision_at_k,recall_at_k,method
0,100,56,1,0.560,0.018117,Baseline
1,250,111,1,0.444,0.035911,Baseline
2,500,180,1,0.360,0.058234,Baseline
3,1000,287,1,0.287,0.092850,Baseline
4,2000,496,1,0.248,0.160466,Baseline
5,100,54,1,0.540,0.017470,TOPSIS
6,250,107,1,0.428,0.034617,TOPSIS
7,500,183,1,0.366,0.059204,TOPSIS
8,1000,293,1,0.293,0.094791,TOPSIS
9,2000,492,1,0.246,0.159172,TOPSIS


In [33]:
# Salvando os resultados da avaliação

test_transaction_counts.to_csv(
    test_output_dir
    / 'test_transaction_counts.csv',
    index=False
)

test_relevant_items.to_csv(
    test_output_dir
    / 'test_relevant_items.csv',
    index=False
)

evaluation_results.to_csv(
    test_output_dir
    / 'evaluation_results.csv',
    index=False
)

evaluation_comparison.to_csv(
    test_output_dir
    / 'evaluation_comparison.csv',
    index=False
)

print('Resultados de teste salvos em:')
print(
    test_output_dir
    / 'evaluation_results.csv'
)

Resultados de teste salvos em:
output/test/evaluation_results.csv


### Relatório

In [34]:
# Relatório geral da execução

print('=' * 70)
print('RELATÓRIO GERAL DA EXECUÇÃO')
print('=' * 70)


# 1. Modo e dados ativos

print('\n[1] MODO E DADOS ATIVOS')

print(
    'Modo:',
    DATA_MODE
)

print(
    'Eventos ativos:',
    f'{len(events_active):,}'
)

print(
    'Visitantes ativos:',
    f"{events_active['visitorid'].nunique():,}"
)

print(
    'Produtos ativos:',
    f"{events_active['itemid'].nunique():,}"
)

print(
    'Tipos de evento:'
)

print(
    events_active['event']
    .value_counts()
    .to_dict()
)

print(
    'Data inicial:',
    events_active['datetime'].min()
)

print(
    'Data final:',
    events_active['datetime'].max()
)


# 2. Divisão temporal

print('\n[2] DIVISÃO TEMPORAL')

print(
    'Data de corte:',
    cutoff_date
)

print(
    'Eventos de treino:',
    f'{len(events_train):,}'
)

print(
    'Eventos de teste:',
    f'{len(events_test):,}'
)

print(
    'Transações no treino:',
    f"{(events_train['event'] == 'transaction').sum():,}"
)

print(
    'Transações no teste:',
    f"{(events_test['event'] == 'transaction').sum():,}"
)


# 3. Grafo

print('\n[3] GRAFO DE TREINO')

print(
    'Visitantes no grafo:',
    f'{train_graph_matrix.shape[0]:,}'
)

print(
    'Produtos no grafo:',
    f'{train_graph_matrix.shape[1]:,}'
)

print(
    'Arestas:',
    f'{train_graph_matrix.nnz:,}'
)

print(
    'Relações únicas visitante-produto:',
    f'{len(user_item_train):,}'
)


# 4. Métricas

print('\n[4] MÉTRICAS DOS PRODUTOS')

print(
    'Produtos com métricas:',
    f'{len(item_metrics_train_final):,}'
)

print('\nResumo estatístico dos critérios:')

display(
    item_metrics_train_final[
        [
            'cooccurrence_score',
            'pagerank',
            'weighted_degree',
            'recency_score',
            'conversion_rate'
        ]
    ]
    .describe()
    .T
)

print('\nValores ausentes:')

display(
    item_metrics_train_final[
        [
            'cooccurrence_score',
            'pagerank',
            'weighted_degree',
            'recency_score',
            'conversion_rate'
        ]
    ]
    .isna()
    .sum()
    .to_frame('quantidade')
)


# 5. Baseline

print('\n[5] TOP 20 DO BASELINE')

display(
    baseline_results_train[
        [
            'baseline_rank',
            'itemid',
            'cooccurrence_score',
            'weighted_degree',
            'pagerank',
            'recency_score',
            'conversion_rate'
        ]
    ]
    .head(20)
)


# 6. TOPSIS

print('\n[6] TOP 20 DO TOPSIS')

display(
    topsis_results_train[
        [
            'topsis_rank',
            'itemid',
            'topsis_score',
            'cooccurrence_score',
            'weighted_degree',
            'pagerank',
            'recency_score',
            'conversion_rate'
        ]
    ]
    .head(20)
)


# 7. Avaliação

print('\n[7] AVALIAÇÃO NO CONJUNTO DE TESTE')

print(
    'Produtos relevantes no teste:',
    f'{len(test_relevant_items):,}'
)

print(
    'Produtos relevantes conhecidos no treino:',
    f'{len(relevant_items):,}'
)

print(
    'Produtos novos no teste:',
    f"{(~test_relevant_items['seen_in_train']).sum():,}"
)

print('\nEvaluation results:')

display(
    evaluation_results
)

print('\nEvaluation comparison:')

display(
    evaluation_comparison
)


print('\n' + '=' * 70)
print('FIM DO RELATÓRIO')
print('=' * 70)

RELATÓRIO GERAL DA EXECUÇÃO

[1] MODO E DADOS ATIVOS
Modo: full
Eventos ativos: 2,756,101
Visitantes ativos: 1,407,580
Produtos ativos: 235,061
Tipos de evento:
{'view': 2664312, 'addtocart': 69332, 'transaction': 22457}
Data inicial: 2015-05-03 03:00:04.384000
Data final: 2015-09-18 02:59:47.788000

[2] DIVISÃO TEMPORAL
Data de corte: 2015-08-18 04:23:31.966000128
Eventos de treino: 2,204,881
Eventos de teste: 551,220
Transações no treino: 17,864
Transações no teste: 4,593

[3] GRAFO DE TREINO
Visitantes no grafo: 1,123,767
Produtos no grafo: 212,916
Arestas: 1,713,175
Relações únicas visitante-produto: 1,713,175

[4] MÉTRICAS DOS PRODUTOS
Produtos com métricas: 212,916

Resumo estatístico dos critérios:


,count,mean,std,min,25%,50%,75%,max
cooccurrence_score,212916.0,558.724070,2479.984134,0.000000e+00,0.000000e+00,2.000000e+00,31.000000,94482.000000
pagerank,212916.0,0.000002,0.000006,1.354114e-07,7.481205e-07,9.475427e-07,0.000002,0.000502
weighted_degree,212916.0,11.207739,35.143769,1.000000e+00,1.000000e+00,3.000000e+00,8.000000,2827.000000
recency_score,212916.0,0.115605,0.191804,9.254453e-03,1.811257e-02,3.803713e-02,0.109472,1.000000
conversion_rate,212916.0,0.004215,0.030434,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1.000000



Valores ausentes:


,quantidade
cooccurrence_score,0
pagerank,0
weighted_degree,0
recency_score,0
conversion_rate,0



[5] TOP 20 DO BASELINE


...


[7] AVALIAÇÃO NO CONJUNTO DE TESTE
Produtos relevantes no teste: 3,292
Produtos relevantes conhecidos no treino: 3,091
Produtos novos no teste: 201

Evaluation results:


,method,k,hits,hit_at_k,precision_at_k,recall_at_k
0,Baseline,5,5,1,1.00,0.001618
1,Baseline,10,9,1,0.90,0.002912
2,Baseline,20,15,1,0.75,0.004853
3,Baseline,50,33,1,0.66,0.010676
4,Baseline,100,56,1,0.56,0.018117
5,TOPSIS,5,1,1,0.20,0.000324
6,TOPSIS,10,5,1,0.50,0.001618
7,TOPSIS,20,12,1,0.60,0.003882
8,TOPSIS,50,32,1,0.64,0.010353
9,TOPSIS,100,54,1,0.54,0.017470



Evaluation comparison:


k     hits        precision_at_k        recall_at_k          
method      Baseline TOPSIS       Baseline TOPSIS    Baseline    TOPSIS
0         5      5.0    1.0           1.00   0.20    0.001618  0.000324
1        10      9.0    5.0           0.90   0.50    0.002912  0.001618
2        20     15.0   12.0           0.75   0.60    0.004853  0.003882
3        50     33.0   32.0           0.66   0.64    0.010676  0.010353
4       100     56.0   54.0           0.56   0.54    0.018117  0.017470


FIM DO RELATÓRIO
